<a href="https://colab.research.google.com/github/kshamasahu706-wq/Customer-Churn-Feature-Engineering/blob/main/Supervised_Classification_Modeling_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Classification / Regression Modeling & Tuning

## Complete Machine Learning Project

**Objective:** Train and compare multiple supervised learning architectures, optimize hyperparameters with `GridSearchCV` and Stratified K-Fold cross-validation, evaluate classification performance, select a champion model, and serialize it for reuse.

### Models implemented
1. Logistic Regression
2. Random Forest
3. XGBoost
4. Support Vector Machine (SVM)

> This notebook implements the **classification** branch of the assignment. The same workflow can be adapted to regression using RMSE, MAE and R².

## 1. Project Overview

Workflow: **Dataset → Train/Test Split → Pipelines → Stratified K-Fold → GridSearchCV → Evaluation → Comparison → Champion Selection → Serialization → Inference**.

The test set is kept separate from hyperparameter optimization to provide an independent final evaluation.

## 2. Problem Statement

Build a robust binary classifier capable of distinguishing the two target classes in a diagnostic dataset. The project should compare several model families rather than relying on one algorithm, tune important hyperparameters, report multiple evaluation metrics, and produce a reusable model artifact.

## 3. Dataset Description

The **Breast Cancer Wisconsin Diagnostic** dataset is loaded from Scikit-Learn. It contains 569 observations, 30 numerical features and two target classes. It is suitable for demonstrating Precision, Recall, F1-Score, ROC-AUC and confusion matrices.

## 4. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report
from xgboost import XGBClassifier

## 5. Load the Dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())

## 6. Initial Data Inspection

In [ ]:
print("Data types:")
display(X.dtypes.value_counts())
print("\nMissing values:")
display(X.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
display(y.value_counts())

## 7. Train/Test Split

The split happens before model selection. `stratify=y` preserves the class distribution in both subsets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 8. Stratified K-Fold Cross-Validation

Five folds are used. Stratification keeps the class distribution more consistent between folds. ROC-AUC is used as the primary validation score for hyperparameter selection.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(cv)

## 9. Preprocessing Strategy

Preprocessing is embedded in each model pipeline. Median imputation handles missing values. Standardization is used for Logistic Regression and SVM. Because these transformations are inside the pipeline, they are fitted only on the training portion of each cross-validation fold.

## 10. Model Architecture — Logistic Regression

Logistic Regression provides a linear baseline. The search tunes regularization strength `C` and the solver.

## 11. Model Architecture — Random Forest

Random Forest is a tree-based ensemble. The search explores tree count, maximum depth and minimum split size.

## 12. Model Architecture — XGBoost

XGBoost is a gradient-boosted tree architecture. The search explores number of estimators, maximum depth and learning rate.

## 13. Model Architecture — SVM

SVM provides a different decision-boundary architecture. The search explores `C` and kernel choice.

## 14. Define Pipelines and Hyperparameter Grids

In [ ]:
models = {
    "Logistic Regression": (
        Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=42))]),
        {"model__C": [0.1, 1, 10], "model__solver": ["liblinear", "lbfgs"]}
    ),
    "Random Forest": (
        Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(random_state=42, n_jobs=-1))]),
        {"model__n_estimators": [200], "model__max_depth": [None, 10], "model__min_samples_split": [2, 5]}
    ),
    "XGBoost": (
        Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1, verbosity=0))]),
        {"model__n_estimators": [100, 200], "model__max_depth": [3, 5], "model__learning_rate": [0.05, 0.1]}
    ),
    "SVM": (
        Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", SVC(probability=True, random_state=42))]),
        {"model__C": [0.5, 1, 10], "model__kernel": ["rbf", "linear"]}
    )
}
print("Models:", list(models))

## 15. Hyperparameter Optimization with GridSearchCV

In [ ]:
searches = {}
rows = []
roc_data = {}

for name, (pipe, param_grid) in models.items():
    print("=" * 70)
    print("Tuning:", name)
    search = GridSearchCV(pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    pred = search.best_estimator_.predict(X_test)
    prob = search.best_estimator_.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    rows.append({"Model": name, "CV ROC-AUC": search.best_score_, "Test Accuracy": accuracy_score(y_test,pred), "Precision": precision_score(y_test,pred), "Recall": recall_score(y_test,pred), "F1-Score": f1_score(y_test,pred), "Test ROC-AUC": auc})
    searches[name] = search
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_data[name] = (fpr,tpr,auc)
    print("Best parameters:", search.best_params_)
    print("Best CV ROC-AUC:", round(search.best_score_,4))

## 16. Model Comparison Table

In [ ]:
comparison = pd.DataFrame(rows).sort_values("CV ROC-AUC", ascending=False).reset_index(drop=True)
display(comparison.style.format({"CV ROC-AUC":"{:.4f}","Test Accuracy":"{:.4f}","Precision":"{:.4f}","Recall":"{:.4f}","F1-Score":"{:.4f}","Test ROC-AUC":"{:.4f}"}))

## 17. Performance Comparison Chart

In [ ]:
comparison.set_index("Model")[["Precision","Recall","F1-Score","Test ROC-AUC"]].plot(kind="bar", figsize=(12,6))
plt.ylim(0,1.05)
plt.ylabel("Score")
plt.title("Tuned Model Performance Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 18. ROC-AUC Curves

The ROC curve shows the relationship between True Positive Rate and False Positive Rate over classification thresholds. AUC summarizes the model's ranking ability.

In [ ]:
plt.figure(figsize=(10,7))
for name,(fpr,tpr,auc_value) in roc_data.items():
    plt.plot(fpr,tpr,linewidth=2,label=f"{name} (AUC = {auc_value:.3f})")
plt.plot([0,1],[0,1],linestyle="--",label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-AUC Curves — Hold-Out Test Set")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 19. Confusion Matrix Comparison

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(11,9))
for ax,(name,search) in zip(axes.ravel(),searches.items()):
    pred=search.best_estimator_.predict(X_test)
    ConfusionMatrixDisplay(confusion_matrix(y_test,pred), display_labels=data.target_names).plot(ax=ax,colorbar=False)
    ax.set_title(name)
plt.suptitle("Confusion Matrices",y=1.02)
plt.tight_layout()
plt.show()

## 20. Precision, Recall and F1-Score

**Precision** measures how many predicted positive cases are correct.

**Recall** measures how many actual positive cases are detected.

**F1-Score** balances Precision and Recall.

In [ ]:
for name,search in searches.items():
    pred=search.best_estimator_.predict(X_test)
    print("="*70)
    print(name)
    print(classification_report(y_test,pred,target_names=data.target_names,digits=4))

## 21. Champion Model Selection

The champion is selected using the highest **mean cross-validated ROC-AUC**. This avoids selecting the model simply because it performed well on the final test set.

In [ ]:
champion_name = comparison.loc[0,"Model"]
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_
print("Champion:",champion_name)
print("Validation ROC-AUC:",round(champion_search.best_score_,4))
print("Best parameters:",champion_search.best_params_)

## 22. Champion Model Final Evaluation

In [ ]:
champion_pred=champion_model.predict(X_test)
champion_prob=champion_model.predict_proba(X_test)[:,1]
champion_metrics=pd.DataFrame({"Metric":["Accuracy","Precision","Recall","F1-Score","ROC-AUC"],"Score":[accuracy_score(y_test,champion_pred),precision_score(y_test,champion_pred),recall_score(y_test,champion_pred),f1_score(y_test,champion_pred),roc_auc_score(y_test,champion_prob)]})
display(champion_metrics)

## 23. Champion Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,champion_pred)
ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=data.target_names).plot(colorbar=False)
plt.title(f"Champion Confusion Matrix — {champion_name}")
plt.tight_layout()
plt.show()

## 24. Champion Classification Report

In [ ]:
print(classification_report(y_test,champion_pred,target_names=data.target_names,digits=4))

## 25. Feature Importance / Coefficients

For a tree-based champion, feature importance is displayed. For a linear champion, coefficient magnitude is displayed.

In [ ]:
final_estimator=champion_model.named_steps["model"]
if hasattr(final_estimator,"feature_importances_"):
    imp=pd.DataFrame({"Feature":X_train.columns,"Importance":final_estimator.feature_importances_}).sort_values("Importance",ascending=False)
    display(imp.head(15))
elif hasattr(final_estimator,"coef_"):
    coef=pd.DataFrame({"Feature":X_train.columns,"Coefficient":final_estimator.coef_[0]})
    coef["Absolute Coefficient"]=coef["Coefficient"].abs()
    display(coef.sort_values("Absolute Coefficient",ascending=False).head(15))
else:
    print("Direct feature importance is not available for this estimator.")

## 26. Inspect Hyperparameter Search Results

In [ ]:
cv_results=pd.DataFrame(champion_search.cv_results_)
display(cv_results[["rank_test_score","mean_test_score","std_test_score","params"]].sort_values("rank_test_score").head(10))

## 27. Serialize the Champion Model

The full fitted pipeline is saved as a `.joblib` file. It contains preprocessing and the tuned estimator, making it reusable for new observations.

In [ ]:
model_path="Supervised_Classification_Champion_Model.joblib"
joblib.dump(champion_model,model_path)
print("Saved:",model_path)
loaded_model=joblib.load(model_path)
print("Reload successful:",type(loaded_model).__name__)

## 28. Reusable Prediction Example

In [ ]:
new_data=X_test.iloc[:5].copy()
loaded_model=joblib.load("Supervised_Classification_Champion_Model.joblib")
pred=loaded_model.predict(new_data)
prob=loaded_model.predict_proba(new_data)[:,1]
display(pd.DataFrame({"Predicted_Class":pred,"Positive_Class_Probability":prob}))

## 29. Leakage Prevention Checklist

- Train/test split is performed before evaluation.
- Test data is never supplied to GridSearchCV.
- Imputation is inside each pipeline.
- Scaling is inside each pipeline.
- Hyperparameters are selected through cross-validation.
- The hold-out test set is reserved for final evaluation.
- The serialized artifact stores the fitted pipeline.

## 30. Reproducibility Checklist

Fixed random states are used for splitting, cross-validation and stochastic models. Compatible installations of Python, NumPy, pandas, scikit-learn, XGBoost, matplotlib and joblib are required.

## 31. Final Model Comparison and Conclusion

In [ ]:
display(comparison)
print(f"Champion selected by validation ROC-AUC: {champion_name}")
print(f"Champion CV ROC-AUC: {champion_search.best_score_:.4f}")

## 32. Final Conclusion

This project satisfies the requested supervised modeling and tuning workflow. Four distinct architectures were trained and compared, hyperparameters were optimized with Stratified 5-Fold GridSearchCV, and the models were evaluated using Precision, Recall, F1-Score, ROC-AUC, ROC curves and confusion matrices.

The champion model was selected using validation ROC-AUC and serialized as a reusable Joblib pipeline. The notebook also includes a reusable inference example and leakage-prevention documentation.

## 33. Submission Deliverables

### Main notebook
`Supervised_Classification_Modeling_Tuning.ipynb`

### Serialized champion model
`Supervised_Classification_Champion_Model.joblib`

### Optional comparison file
`Supervised_Model_Comparison.csv`

These files provide the requested proof of implementation, comparison, evaluation and serialization.

## 34. Assignment Summary

**Task:** Supervised Classification / Regression Modeling & Tuning

**Dataset:** Breast Cancer Wisconsin Diagnostic

**Architectures:** Logistic Regression, Random Forest, XGBoost and SVM

**Optimization:** GridSearchCV with Stratified 5-Fold Cross-Validation

**Metrics:** Accuracy, Precision, Recall, F1-Score and ROC-AUC

**Visual proof:** ROC-AUC curves and confusion matrices

**Champion selection:** Highest validation ROC-AUC

**Deployment artifact:** Serialized `.joblib` pipeline